# Train RayenFD iTransformer on GPU

This notebook trains `itransformer_rayenfd`, the Rayen iTransformer variant whose demand plane is pinned to `net_demand` from the last input step, i.e. `nD(t-1)`. It then reports only the requested metrics:

- regular test: `WAPE`, `n_ramp_violations`, `n_netdemand_balance_violations`, `SOC_possible_starting`
- simulated test: `n_ramp_violations`, `n_netdemand_balance_violations`, `SOC_possible_starting`

The simulated test is the closed-loop rollout over the TEST-split stress episodes in `constraints/stress_episodes.json`; WAPE is intentionally not reported for it.

In [ ]:
# Colab setup. Run this from the repository root after cloning/uploading the repo.
!pip -q install pyarrow scikit-learn pandas numpy

In [ ]:
from pathlib import Path
import os

REPO = Path.cwd()
if not (REPO / "ml" / "train_hist.py").exists():
    raise RuntimeError(
        "Notebook is not running from the repo root. In Colab, clone/upload "
        "energy_modelling, then %cd into that folder before continuing."
    )

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/energy_runs/rayenfd_itransformer"
except Exception:
    OUT = str(REPO / "colab_runs" / "rayenfd_itransformer")

Path(OUT).mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("output:", OUT)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is available. In Colab: Runtime > Change runtime type > GPU.")
print("CUDA:", torch.cuda.get_device_name(0))

In [ ]:
# Train or resume. train_hist.py checkpoints every epoch, so rerunning this cell resumes safely.
!python ml/train_hist.py --arch itransformer_rayenfd --seed 0 --device cuda --out "$OUT"

In [ ]:
# Evaluate requested metrics only.
METRICS_OUT = f"{OUT}/rayenfd_requested_metrics.json"
!python constraints/eval_rayenfd_requested_metrics.py --ckpt-dir "$OUT" --seed 0 --device cuda --out "$METRICS_OUT"

In [ ]:
import json
import pandas as pd

with open(METRICS_OUT) as f:
    metrics = json.load(f)

regular_cols = [
    "WAPE",
    "n_ramp_violations",
    "n_netdemand_balance_violations",
    "SOC_possible_starting",
]
sim_cols = [
    "n_ramp_violations",
    "n_netdemand_balance_violations",
    "SOC_possible_starting",
]

regular = pd.DataFrame([metrics["regular_test"]], index=["regular_test"])[regular_cols]
simulated = pd.DataFrame([metrics["simulated_test"]], index=["simulated_test"])[sim_cols]

display(regular)
display(simulated)